In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from google.colab import files
import io # Added import for io

!pip -q install openpyxl statsmodels

uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_excel(io.BytesIO(uploaded[file_name])) # Changed to pd.read_excel and used io.BytesIO

# --- chuẩn hóa ---
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df["Country"] = df["Country"].astype(str)

id_cols = ["Year", "Country"]
value_cols = [c for c in df.columns if c not in id_cols]

for c in value_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.sort_values(["Country", "Year"]).reset_index(drop=True)

fill_years = [2022, 2023, 2024,2025]
countries = df["Country"].unique()

# --- thêm dòng nếu thiếu ---
existing = set(zip(df["Country"], df["Year"].astype("Int64")))
need = []
for country in countries:
    for y in fill_years:
        if (country, y) not in existing:
            need.append({"Country": country, "Year": y})

if need:
    add_rows = pd.DataFrame(need)
    for c in value_cols:
        add_rows[c] = np.nan
    df = pd.concat([df, add_rows], ignore_index=True)

df = df.sort_values(["Country", "Year"]).reset_index(drop=True)

# --- forecast Holt 1 bước ---
def holt_forecast_one_step(y: np.ndarray):
    if len(y) < 4:
        return np.nan, np.nan, np.nan

    # nếu gần như hằng số -> trả last value
    if np.nanstd(y) == 0:
        return float(y[-1]), np.nan, np.nan

    try:
        model = ExponentialSmoothing(
            y,
            trend="add",              # Holt add trend
            seasonal=None,
            initialization_method="estimated"
        )
        fit = model.fit(optimized=True)
        forecast_val = float(fit.forecast(1)[0])
        alpha = fit.params['smoothing_level']
        beta = fit.params['smoothing_trend']
        return forecast_val, alpha, beta
    except:
        return np.nan, np.nan, np.nan

# --- fill lấp dần ---
holt_params = [] # List to store Holt parameters
for country in df["Country"].unique():
    c_mask = (df["Country"] == country)

    for year in fill_years:
        idx_list = df.index[c_mask & (df["Year"] == year)]
        if len(idx_list) == 0:
            continue
        idx = idx_list[0]

        for col in value_cols:
            if pd.notna(df.at[idx, col]):
                continue

            hist = df.loc[c_mask & (df["Year"] < year), col].dropna().values
            fc, alpha, beta = holt_forecast_one_step(hist) # Unpack the returned tuple
            if pd.notna(fc):
                df.at[idx, col] = fc
                # Store parameters only when a forecast was actually made and applied
                holt_params.append({
                    "Country": country,
                    "Year": year,
                    "Variable": col,
                    "Alpha": alpha,
                    "Beta": beta
                })

# Create a DataFrame for Holt parameters
holt_params_df = pd.DataFrame(holt_params)

# --- export df with forecasts ---
out_df = "HOLT TREND 9.4.xlsx"
df.to_excel(out_df, index=False)
files.download(out_df)

Saving DATASET 17 NƯỚC MỚI - Copy.xlsx to DATASET 17 NƯỚC MỚI - Copy.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>